In [ ]:
import matplotlib.pyplot as plt
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
import cv2

In [ ]:
image_paths = './births/' + pd.Series(os.listdir('./births'))

In [ ]:
df = pd.DataFrame(list(image_paths),columns=['image_path'])
df['image'] = df.image_path.map(Image.open)

In [ ]:
def split_name(image_path):
    name = ''.join(image_path.split('/')[-1])[0:-4].split('-')
    return pd.Series([
        int(name[0]),
        int(name[1]),
        int(name[2]),
    ])
df[['gid', 'sgid', 'fileid']] = df.image_path.apply(split_name)

In [ ]:
def get_resolution(img):
    w,h = img.size
    return pd.Series([w,h])
df[['width', 'height']] = df.image.apply(get_resolution)
df['resolution'] = df.width.astype(str) + 'x' + df.height.astype(str)
print('Unique resolutions:', df.resolution.unique())
df.resolution.value_counts().head(5)

In [ ]:
# Filter

df = df[df.resolution.eq('4100x3148')]
df = df.reset_index(drop=True)

In [ ]:
def crop_and_split_pages(img,verbose=False):
    gray_image = cv2.cvtColor(np.array(img), cv2.COLOR_BGR2GRAY)
    img_corners = cv2.adaptiveThreshold(
        gray_image, 
        255,                                  # Wartość dla tła (255 = idealna biel)
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,       # Metoda liczenia wagi (Gaussian jest płynniejszy)
        cv2.THRESH_BINARY,                    # Tryb: tekst na czarno, tło na biało
        3,                                   # Rozmiar bloku (musi być nieparzysty, np. 11, 21, 31)
        10                                    # Stała odejmowana od średniej
    )
    img_corners = cv2.GaussianBlur(img_corners, (5,5), 0)
    img_corners = 255 - img_corners

    if verbose:
        plt.imshow(img_corners)
        plt.show()

    hcut = (np.array(img_corners)).mean(axis=0)
    hcut_mean = np.mean(hcut)
    hcut_idxs = (hcut>hcut_mean).nonzero()[0]
    hcut1,hcut2 = hcut_idxs[0], hcut_idxs[-1]

    vcut = (np.array(img_corners)).mean(axis=1)
    vcut_mean = np.mean(vcut)
    vcut_idxs = (vcut>vcut_mean).nonzero()[0]
    vcut1,vcut2 = vcut_idxs[0], vcut_idxs[-1]
    # plt.plot(vcut)
    # plt.show()
    # plt.plot(hcut)
    # plt.show()
    hcut1,vcut1,hcut2,vcut2 = hcut1-30,vcut1-30,hcut2+30,vcut2+30 # Margins
    # vcut1 += 53 # Remove heading

    img = img.crop((hcut1,vcut1,hcut2,vcut2))
    page1 = img.crop((0,0,img.width//2,img.height))
    page2 = img.crop((img.width//2,0,img.width,img.height))
    return page1,page2


In [ ]:
def get_peaks(hgram,w1=5,w2=10,dillute=151,verbose=False):
    hgram_rmean1 = np.convolve(hgram, np.ones(w1) / w1, mode='same')
    hgram_rmean2 = np.convolve(hgram, np.ones(w2) / w2, mode='same')

    boost = hgram_rmean1-hgram_rmean2
    # boost[0:100] = 0
    # boost[-100::] = 0

    if verbose:
        plt.plot(boost)
        plt.show()

    maxed = torch.nn.functional.max_pool1d(torch.tensor([boost]),dillute,1,dillute//2)[0]
    boost_maxed = torch.tensor(boost)
    boost_maxed[(1-(boost==maxed).long()).bool()] = 0
    
    if verbose:
        plt.plot(boost_maxed)
        plt.show()
    # boost_maxed_nonzero = boost_maxed[boost_maxed!=0]
    # plt.imshow(img)
    # for c in peaks:
    #     plt.hlines(xmin=0,xmax=img.width-1,y=c)
    # plt.title(f'w1: {w1}, w2: {w2}')
    # plt.show()
    return (boost_maxed>np.median(hgram)).nonzero()

def get_h_conturs(img,verbose=False):
    gray_image = cv2.cvtColor(np.array(img), cv2.COLOR_BGR2GRAY)
    img_corners = cv2.adaptiveThreshold(
        gray_image, 
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        3,
        5
    )
    # img_corners = cv2.GaussianBlur(img_corners, (13,13), 0)

    if verbose:
        plt.imshow(img_corners)
        plt.show()

    thresh = cv2.adaptiveThreshold(img_corners, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                    cv2.THRESH_BINARY_INV, 7, 5)

    kernel_length = np.array(img).shape[1] // 160
    # vert_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, kernel_length))
    hori_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_length, 1))

    # img_vw = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vert_kernel, iterations=2)
    img_hw = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, hori_kernel, iterations=2)

    img_hw = cv2.GaussianBlur(img_hw, (25,25), 13)
    if verbose:
        plt.imshow(img_hw)
        plt.show()
    return img_hw

def draw_lines(img,lines):
    plt.imshow(img)
    end_lines = []
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line
            # if abs(np.rad2deg(np.arctan2(y1 - y2, x2 - x1))) < 4:
            plt.plot((x1, x2), (y1, y2),color='r')
            end_lines.append((x1,y1,x2,y2))
    print(end_lines)
    print(len(end_lines))
    plt.show()

def merge_lines(lines, dist_threshold=50):
    if lines is None: return []

    processed_lines = []
    for line in lines:
        x1, y1, x2, y2 = line
        mid_x, mid_y = (x1 + x2) / 2, (y1 + y2) / 2
        processed_lines.append({'coords': (x1, y1, x2, y2), 'mid': (mid_x, mid_y), 'used': False})

    merged_lines = []

    for i in range(len(processed_lines)):
        if processed_lines[i]['used']: continue
        
        # Startujemy nową grupę
        current_group = [processed_lines[i]['coords']]
        processed_lines[i]['used'] = True
        
        for j in range(i + 1, len(processed_lines)):
            if processed_lines[j]['used']: continue

            # Sprawdź odległość między środkami linii
            dist = abs(processed_lines[i]['mid'][1] - processed_lines[j]['mid'][1])
            
            if dist < dist_threshold:
                current_group.append(processed_lines[j]['coords'])
                processed_lines[j]['used'] = True
        
        # Uśrednij współrzędne dla grupy (uproszczony model)
        avg_coords = np.mean(current_group, axis=0).astype(int)
        merged_lines.append(avg_coords)
        
    return merged_lines

In [ ]:
def get_rows(page,verbose=False):
    img_hw = get_h_conturs(page,verbose=verbose)

    min_length = 500
    lines = cv2.HoughLinesP(img_hw, 1, np.pi/180, threshold=250, 
                            minLineLength=min_length, maxLineGap=700)[:,0].copy() # .copy() needed for memory leaks

    lines = lines[abs(np.rad2deg(np.arctan2(lines[:,1] - lines[:,3], lines[:,2] - lines[:,0])))<4]

    if verbose:
        draw_lines(img_hw,lines)

    lines_m =merge_lines(lines,dist_threshold=100)
    if verbose:
        draw_lines(img_hw,lines_m)
    rows = np.array([int((y1+y2)/2) for (x1,y1,x2,y2) in lines_m])
    rows.sort()

    return rows

In [ ]:
from pathlib import Path
import random
from tqdm import tqdm
import shutil

rows_imgs = []
rows_lab = []

temp_dir = Path("rows_temp")

if temp_dir.exists():
    shutil.rmtree(temp_dir)
temp_dir.mkdir(exist_ok=True)


rows_count = 0
t = tqdm(range(len(df[0:100])))
for i in t:
    sample = df.loc[i]
    page1,page2 = crop_and_split_pages(sample.image)

    for page_side in [0,1]:
        p = page1 if page_side==0 else page2
        rows = get_rows(p,verbose=False)

        # plt.imshow(p)
        # for y in rows:
        #     plt.hlines(xmin=0,xmax=p.width-1,y=y)
        # # for x in cols:
        # #     plt.vlines(ymin=0,ymax=img.height-1,x=x)
        # plt.show()

        margin=10
        for i in range(0,len(rows)-1):
            rows_count += 1

            row_img = p.crop((0,int(rows[i])-margin,p.width-1,int(rows[i+1])+margin)).copy()
            row_img = row_img.crop((0,0,1738,400))

            filename = temp_dir / f'{random.randint(100000000000,999999999999)}.jpg'
            row_img.save(filename)

            rows_imgs.append(Image.open(filename))
            rows_lab.append({'source': sample.image_path, 'page_side': page_side, 'idx':i})
            
            # plt.imshow(rows_ds[-1])
            # plt.show()
        t.set_postfix_str(f'Rows: {rows_count}')
            
# Load to numpy without breaking lazy-loading
rows_imgs_numpy = np.empty(len(rows_imgs), dtype=object)
for i in range(len(rows_imgs)):
    rows_imgs_numpy[i] = rows_imgs[i]
rows_imgs = rows_imgs_numpy

rows_lab = np.array(rows_lab)

In [ ]:
plt.hist(np.array([img.height for img in rows_imgs if img.height<700]),bins=150)

# Outliers detection

In [ ]:
# height_outliers = np.array([img.height>400 for img in rows_imgs])

# rows_imgs = rows_imgs[height_outliers==False]
# rows_lab = rows_lab[height_outliers==False]

In [ ]:
from transformers import AutoImageProcessor, AutoModel
from PIL import Image
import torch

processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small")
model = AutoModel.from_pretrained("facebook/dinov2-small")

vectors = torch.Tensor()

for image in tqdm(rows_imgs,desc='Extracting features with DINOv2-small'):
    inputs = processor(images=image, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)
        vectors = torch.cat([vectors,outputs.last_hidden_state[:, 0, :]],dim=0)
vectors = vectors.numpy()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from sklearn.cluster import KMeans

# --- Assume `image_vectors` is a list/array of your extracted DINOv2 features ---
# For this example, I generated synthetic data: 300 normal vectors, 15 random outliers
np.random.seed(42)
X = vectors

# 1. Detect Anomalies using Isolation Forest
# Contamination is the estimated percentage of outliers in your dataset (e.g., 5%)
iso_forest = IsolationForest(contamination=0.05, random_state=42)
preds = iso_forest.fit_predict(X)

# Get anomaly scores (lower score = more anomalous)
scores = iso_forest.decision_function(X)

# Get the array indices of the detected outliers
outlier_indices = np.where(preds == -1)[0]
inlier_indices = np.where(preds == 1)[0]

# 2. Dimensionality Reduction for Visualization (384D -> 2D)
pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X)

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_2d = tsne.fit_transform(X_pca)

# 3. Plotting
plt.figure(figsize=(10, 8))

# Plot the normal cluster
plt.scatter(X_2d[inlier_indices, 0], X_2d[inlier_indices, 1], 
            c='#3498db', label='Normal Images', alpha=0.7, edgecolors='w', s=60)

# Plot the outliers
plt.scatter(X_2d[outlier_indices, 0], X_2d[outlier_indices, 1], 
            c='#e74c3c', label='Detected Outliers', marker='X', s=100)



plt.title('t-SNE Visualization of Image Embeddings')
plt.legend()
plt.show()



In [ ]:
n_clusters = 3

km = KMeans(n_clusters)
cpred = km.fit_predict(X)

def plot_kmeans_tsne(X_2d, labels, images, sample_fraction=0.7):
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Use a colormap for the clusters
    cmap = plt.cm.get_cmap('tab10', n_clusters)
    
    # Draw background dots colored by cluster
    scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap=cmap, alpha=0.3, s=60)
    
    # Overlay images with a border matching their cluster color
    n_images = len(images)
    indices = np.random.choice(range(n_images), max(1, int(n_images * sample_fraction)), replace=False)

    for i in indices:
        # Process thumbnail
        img = images[i].convert("RGB")
        img.thumbnail((48, 48), Image.Resampling.LANCZOS)
        img_np = np.array(img)

        # Get the color assigned to this cluster
        cluster_color = cmap(labels[i])

        # Create the image box with a colored frame
        imagebox = OffsetImage(img_np, zoom=0.6)
        ab = AnnotationBbox(
            imagebox, (X_2d[i, 0], X_2d[i, 1]),
            xycoords='data',
            frameon=True,
            bboxprops=dict(edgecolor=cluster_color, linewidth=2.5) # Color border by cluster
        )
        ax.add_artist(ab)

    plt.colorbar(scatter, ticks=range(n_clusters), label='Cluster ID')
    plt.title(f'K-Means Clustering (k={n_clusters}) on DINOv2 Embeddings', fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.1)
    plt.show()

# Run the new visualization
plot_kmeans_tsne(X_2d, cpred, rows_imgs)
print(np.bincount(cpred))
normal_class = np.bincount(cpred).argmax()
print('Normal class:', normal_class)

In [ ]:
dino_outlier = cpred[i]!=normal_class 

rows_imgs = rows_imgs[dino_outlier==False]
rows_lab = rows_lab[dino_outlier==False]